# Phase 2 Baseline Replay: Real Historical Sequence
This notebook evaluates Persistence and PySTEPS baselines on a real historical sequence (July 25, 2023).

In [ ]:
import sys
import numpy as np
import pandas as pd
import zarr
import matplotlib.pyplot as plt
import json
from pathlib import Path
from matplotlib.colors import LogNorm
from matplotlib import animation
from IPython.display import HTML, display

ROOT = Path.cwd().parent
sys.path.append(str(ROOT / 'src'))
CUBES_DIR = ROOT / 'data' / 'processed' / 'cubes'
REPORTS_DIR = ROOT / 'reports'

## Load Metrics from Evaluation Run
We generated these metrics using `scripts/run_baseline_eval.py`.

In [ ]:
metrics_file = REPORTS_DIR / 'phase2_eval_metrics.json'
with open(metrics_file, 'r') as f:
    report = json.load(f)

pers_df = pd.DataFrame(report['baselines']['persistence'])
pysteps_df = pd.DataFrame(report['baselines']['pysteps'])

display(pers_df.head())
display(pysteps_df.head())

## Visualizing F1-Score Decay over Lead Time
We plot how F1 score (threshold 1.0 mm/h) decays as prediction horizon increases.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(pers_df['lead_time'] * 30, pers_df['f1_1.0'], marker='o', label='Persistence')
plt.plot(pysteps_df['lead_time'] * 30, pysteps_df['f1_1.0'], marker='x', label='PySTEPS')
plt.xlabel('Lead Time (Minutes)')
plt.ylabel('F1 Score (Threshold > 1.0 mm/h)')
plt.title('F1 Score Decay over Lead Time (Real Data)')
plt.legend()
plt.grid(True)
plt.show()

## Visual Replay: Actual vs Persistence vs PySTEPS
Here we take a specific timeframe in our sequence (t=10) and visualize the +30, +60, +90, and +120 minute predictions alongside the error maps.

In [ ]:
from jalrakshak_ml.nowcast.persistence import PersistenceNowcast
from jalrakshak_ml.nowcast.pysteps_adapter import PystepsNowcast

weather_path = CUBES_DIR / 'weather.zarr'
root = zarr.open(str(weather_path), mode='r')
gpm = root['rainfall_gpm'][:]
times = root['time'][:]

# Select an interesting frame t=10
t = 10
history_length = 3
lead_times = 4 # +30, +60, +90, +120

history = gpm[t-history_length:t]
future_obs = gpm[t:t+lead_times]

pers_model = PersistenceNowcast()
pysteps_model = PystepsNowcast()

pers_pred = pers_model.predict(history[-1], lead_times)
pysteps_pred = pysteps_model.predict(history, lead_times)

# Prepare for plotting
fig, axes = plt.subplots(lead_times, 5, figsize=(20, 15))

for i in range(lead_times):
    lead_min = (i + 1) * 30
    obs = future_obs[i]
    p_pred = pers_pred[i]
    ps_pred = pysteps_pred[i]
    
    p_err = p_pred - obs
    ps_err = ps_pred - obs
    
    vmin, vmax = 0.1, 50
    
    ax = axes[i, 0]
    im = ax.imshow(obs, cmap='Blues', norm=LogNorm(vmin=vmin, vmax=vmax))
    ax.set_title(f'Actual (+{lead_min}m)')
    ax.axis('off')
    
    ax = axes[i, 1]
    ax.imshow(p_pred, cmap='Blues', norm=LogNorm(vmin=vmin, vmax=vmax))
    ax.set_title(f'Persistence (+{lead_min}m)')
    ax.axis('off')
    
    ax = axes[i, 2]
    ax.imshow(ps_pred, cmap='Blues', norm=LogNorm(vmin=vmin, vmax=vmax))
    ax.set_title(f'PySTEPS (+{lead_min}m)')
    ax.axis('off')
    
    # Error maps (-10 to 10 mm/h)
    err_lim = 10
    ax = axes[i, 3]
    ax.imshow(p_err, cmap='coolwarm', vmin=-err_lim, vmax=err_lim)
    ax.set_title(f'Persistence Error (+{lead_min}m)')
    ax.axis('off')
    
    ax = axes[i, 4]
    ax.imshow(ps_err, cmap='coolwarm', vmin=-err_lim, vmax=err_lim)
    ax.set_title(f'PySTEPS Error (+{lead_min}m)')
    ax.axis('off')

plt.tight_layout()
plt.show()

## Chronological Event Animation
Let's animate the raw GPM IMERG sequence directly to observe the storm evolution natively.

In [ ]:
fig_anim, ax_anim = plt.subplots(figsize=(6, 6))
ax_anim.axis('off')

vmin, vmax = 0.1, 50
im_anim = ax_anim.imshow(np.zeros_like(gpm[0]), cmap='Blues', norm=LogNorm(vmin=vmin, vmax=vmax))
title = ax_anim.set_title('')

def update(frame):
    im_anim.set_array(gpm[frame])
    title.set_text(f'GPM IMERG: {times[frame]}')
    return [im_anim, title]

anim = animation.FuncAnimation(fig_anim, update, frames=len(gpm), interval=300, blit=True)
plt.close(fig_anim) # Prevent duplicate static plot
HTML(anim.to_jshtml())

## Conclusion
Both baselines are fully functional on real historical data. The visual error maps indicate PySTEPS captures advection better than persistence, although it struggles with growth/decay of convective cells. 

**WE ARE READY FOR PHASE 3: DEEP LEARNING!**